# Visual Analytics

## Assignment 3

**Instructor:** Dr. Marco D'Ambros  
**TAs:** Giuseppe Crupi, Mattia Giannaccari

**Contacts:** marco.dambros@usi.ch, giuseppe.crupi@usi.ch, mattia.giannaccari@usi.ch

**Due Date:** May 25, 2026 @ 23:55

---
The goal of this assignment is to use **Spark (PySpark)** and **Polars** in Jupyter notebooks.  
The files `trip_data.csv`, `trip_fare.csv`, and `nyc_boroughs.geojson` are available in the provided folder: [Assignment3-data](https://usi365-my.sharepoint.com/:f:/g/personal/armenc_usi_ch/Ejp7sb8QAMROoWe0XUDcAkMBoqUFk-w2Vgroup025NhAww?e=2I7SMC).

- Use **Spark** to solve **Exercises 1–4**
- Use **Polars** to solve **Exercises 5–8**

Please name your notebook file as `SurnameName_Assignment3.ipynb`

# ⚡️ Spark Exercises (50 pts)

### Initial Spark Setup

In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Assignment3") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 16:28:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



### Importing and cleaning the data

In [2]:
# Loading the datasets
# After selecting the dataset thanks to the related paths, header=True is used to consider the first row as column names while 
# inferSchema=True is used to automatically detect data types

trip_data = spark.read.csv(
    "data-assignment3/trip_data.csv", header=True, inferSchema=True
)

trip_fare = spark.read.csv(
    "data-assignment3/trip_fare.csv", header=True, inferSchema=True
)

# Since the column names have spaces, they need to be trimmed before joining
trip_data = trip_data.toDF(*[c.strip() for c in trip_data.columns])
trip_fare = trip_fare.toDF(*[c.strip() for c in trip_fare.columns])

# Visualizing the clean datasets without spaces in column names
trip_data.show(5)
trip_fare.show(5)

+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+
|           medallion|        hack_license|vendor_id|rate_code|store_and_fwd_flag|    pickup_datetime|   dropoff_datetime|passenger_count|trip_time_in_secs|trip_distance|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|
+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+
|89D227B655E5C82AE...|BA96DE419E711691B...|      CMT|        1|                 N|2013-01-01 15:11:48|2013-01-01 15:18:10|              4|              382|          1.0|      -73.978165|      40.757977|       -73.989838|       40.751171|
|0BD7C8F5BA12B88E0...|9FD8F69F0804BDB55...| 

### Exercise 1 (8 pts)
Join the `trip_data` and `trip_fare` dataframes into one, considering only trips from January 1–7, 2013. Filter out trips where the total amount charged is 0 or less, or the trip distance is 0 or less. Report how many rows are removed by each filter, the total number of rows after filtering, and the average tip amount across the resulting dataset.

### Solution 1

#### Merging the two dataframes

In [5]:
# Merging the datasets on the selected columns: medallion, hack_license, pickup_datetime
# This is because the medallion and hack_license columns identify the taxi and the driver, 
# while the pickup_datetime column identifies the specific trip.
df = trip_data.join(
    trip_fare,
    on=["medallion", "hack_license", "pickup_datetime"],
    how="inner"
)

print(f"Rows after merging: {df.count()}")

Rows after merging: 14776615


#### Filtering trips from Jan 1 2013 to Jan 7 2013

In [ ]:
rows_before = df.count()

# Filter trips between January 1 and January 7, 2013
df = df.filter(
    (F.col("pickup_datetime") >= "2013-01-01") &
    (F.col("pickup_datetime") <  "2013-01-08")
)

rows_after = df.count()

# Printing the number of rows removed after the date filtering
print(f"Rows removed after date filtering: {rows_before - rows_after}")

In [ ]:
rows_before = df.count()

# Filter trips with total_amount <= 0
df = df.filter(F.col("total_amount") > 0)

rows_after = df.count()

# Printing the number of rows removed after the total_amount filtering
print(f"Rows removed by total_amount filter: {rows_before - rows_after}")

In [ ]:
# Cell 5 - Filter 2: trip_distance <= 0
rows_before = df.count()
df = df.filter(F.col("trip_distance") > 0)
removed = rows_before - df.count()
avg_tip = df.agg(F.avg("tip_amount")).collect()[0][0]

print(f"Rows removed by trip_distance filter: {removed}")
print(f"Total rows after all filters:         {df.count()}")
print(f"Average tip amount:                   ${avg_tip:.2f}")

### Exercise 2 (12 pts)
For each hour of the day (0–23), compute the average trip duration in minutes and the average trip distance. Provide a graphical representation that allows comparing both metrics across hours side by side. You may want to have a look at: https://docs.bokeh.org/en/latest/docs/user_guide/basic/bars.html#grouping

### Solution 2

### Exercise 3 (14 pts)
Consider only the boroughs Queens, Staten Island, and EWR. Create a dataframe that shows, for each payment type, the total fare amount collected for trips *originating from* each of those three boroughs, broken down by *destination borough* (including all boroughs as destinations).

> For example, for Queens you should consider:
> - Queens → Queens (cash), Queens → Queens (card), ...
> - Queens → Manhattan (cash), Queens → Manhattan (card), ...
> - and so on for all destination boroughs.


### Solution 3

### Exercise 4 (16 pts)
Create a dataframe where each row represents a driver, and there is one column per hour of the day (0–23). For each driver-hour, the dataframe provides the maximum number of consecutive trips where the tip amount was strictly greater than $0.

> For example, if for driver B we have trips starting in hour 14 (sorted by pickup time):
>
> - Trip 1: tip = $2.00
> - Trip 2: tip = $0.00
> - Trip 3: tip = $1.50
> - Trip 4: tip = $3.00
>
> The longest streak of tipped trips in hour 14 is 2 (Trips 3 and 4).

Additionally, print the pair (driver, hour) with the maximum streak.

### Solution 4

# 🐻‍❄️ Polars Exercises (50 pts)

In this section, you will use **Polars** to perform data cleaning, transformation, and analysis on the NYC taxi dataset.

You will work with the merged dataset obtained from:
- `trip_data.csv`
- `trip_fare.csv`

### Exercise 5 (10 pts)

Perform a sequence of data cleaning steps on the dataset:

1. Remove trips where:
   - `trip_distance <= 10` but `fare_amount > 100`.
   - `trip_distance > 100` or `trip_distance <= 0>` miles.

2. Remove trips with:
   - missing timestamps (`pickup_datetime`, `dropoff_datetime`).
   - `dropoff_datetime <= pickup_datetime`.

After each step:
- Report how many rows were removed.

Finally:
- Report the number of remaining rows.
- Check whether duplicate records exist (based on `medallion`, `hack_license`, `pickup_datetime`).

### Solution 5

### Exercise 6 (12 pts)

Analyze temporal patterns in taxi demand:

1. Group the data by `(weekday, hour)` and compute:
   - total number of trips
   - average fare per trip

2. Visualize the results using a **heatmap**

3. Return the top 5 `(weekday, hour)` by average fare.

### Solution 6

### Exercise 7 (12 pts)

Define a *high-value trip* as one satisfying **at least two** of the following conditions:

- `fare_amount` is in the top 10%
- `tip_amount > 50%` of `fare_amount`
- `trip_distance < 2 miles` AND `fare_amount` above the median

Tasks:

1. Extract all high-value trips.
2. Select only the rides longer than 10 miles (in a straight line).
3. Report the total number of such trips.
4. Create a scatterplot:
   - x-axis: `trip_distance`
   - y-axis: `fare_amount`

4. Briefly interpret the observed patterns

### Solution 7

### Exercise 8 (16 pts)

Analyze driver performance using earnings efficiency:

1. For each trip, compute:
   - trip duration in hours.
   - total earnings = `fare_amount + tip_amount`.

2. Filter:
   - only keep durations between `3 minutes and 5 hours`.

3. For each driver (`hack_license`), compute:
   - total earnings.
   - total driving time (in hours).
   - earnings per hour.
   - total number of trips.

4. Select the **top 15% drivers** based on number of trips.

5. Classify trips into:
   - **day** (i.e., `06:00–18:00`).
   - **night** (remaining hours).

6. Compare driver efficiency:
   - Plot the distribution of earnings per hour for `day vs night drivers` (notice that a driver can be both a "day" and "night" driver in case it performed at least one day ride and one night ride).

7. Answer:
   - Which group appears more efficient?
   - Provide a short explanation based on your results.

### Solution 8